# Session 7 · Case study: validating and tuning a Berlin price model

*Author: course team · Licence: CC-BY-4.0*

**Question.** What is a Berlin Airbnb listing worth per night, and how well would a price model work for a host who lists a flat for the first time? A good answer needs an honest validation (Part A) and a sensibly tuned model (Part B).

| Part | Block | Task |
|---|---|---|
| A | 0:00–0:45 | Cross-validate the price model with random and host-grouped folds; report the error in euros with a bootstrap interval |
| B | 1:00–1:45 | Regularise with ridge and lasso on 480 columns, draw validation and learning curves, tune with grid and randomised search |
| C | 2:00–2:45 | Demonstrate two leaking workflows (target encoding fitted on all rows; the same host in training and validation), fix them with a Pipeline and grouped folds, and check the scores on new hosts |

**Data.** Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0), prepared with `uv run python case-study/prepare_airbnb.py`. The data were collected from public listing pages; the course copy contains no host names. Use `host_id` only as a grouping key and report results in aggregate.

Tasks for you are marked **Task**. The whole notebook runs in about three minutes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
AIRBNB = ROOT / "case-study" / "data" / "airbnb"
import json

from scipy.stats import loguniform, randint
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, GroupShuffleSplit, KFold, RandomizedSearchCV,
                                     cross_val_predict, learning_curve, validation_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

## The price-model table

Listings with a minimum stay of 28 nights or more are medium-term rentals; their advertised nightly prices follow another basis (look at the medians below), so they belong to a separate model. The 26 listings priced below €10 or above €1,000 are extreme outliers (Session 4) and are left out; the remaining 6,675 listings are the short-stay table of Sessions 4–6 minus these outliers.

In [ ]:
listings = pd.read_parquet(AIRBNB / "listings.parquet")
# short-stay listings with a price (minimum stay < 28 nights, as in Sessions 4-6),
# without the 26 outliers priced below EUR 10 or above EUR 1,000
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
print(len(listings), "listings in the snapshot,", len(bnb), "in the price-model table,",
      bnb["host_id"].nunique(), "hosts")
print(listings.groupby(listings["minimum_nights"] >= 28)["price"].median().rename({False: "< 28 nights", True: ">= 28 nights"}))
print(bnb.groupby("room_type")["price"].agg(["count", "median"]).round(0))

Features: size, location (distance to Alexanderplatz, district, neighbourhood), property type, reviews, availability and 257 amenity indicators. The target is the log of the price: errors on the log scale are relative errors, and the model's predictions are turned back into euros with `np.exp`.

In [ ]:
mlb = MultiLabelBinarizer()
amen = pd.DataFrame(mlb.fit_transform(bnb["amenities"].map(json.loads)), columns=["am_" + a for a in mlb.classes_])
amen = amen.loc[:, amen.sum() >= 20]                       # amenities listed by at least 20 listings
num = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "minimum_nights",
       "availability_365", "number_of_reviews", "review_scores_rating"]
cat = ["room_type", "district", "neighbourhood", "property_type"]
X = pd.concat([bnb[num + cat], amen], axis=1)
y = np.log(bnb["price"])
hosts = bnb["host_id"]
prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat),
    (StandardScaler(), list(amen.columns)))
print(X.shape, "->", prep.fit_transform(X).shape[1], "model columns")
n_per_host = hosts.map(hosts.value_counts())
print("share of listings whose host has several listings:", round((n_per_host > 1).mean(), 3))

## Part A · Honest cross-validation

### A1 · Random versus host-grouped folds

Four models: the small linear model of Session 6 (guests, room type, distance, district), ridge regression on all 480 columns, a random forest and gradient boosting (Session 10; here black boxes). Each is cross-validated with random `KFold` and with `GroupKFold` by host. We report R² on the log scale and the mean absolute error (MAE) in euros.

In [ ]:
def euro_scores(model, Xm, splitter, groups=None):
    """Cross-validated predictions -> R² (log scale) and MAE in euros."""
    pred = cross_val_predict(model, Xm, y, cv=splitter, groups=groups, n_jobs=-1)
    return r2_score(y, pred), mean_absolute_error(bnb["price"], np.exp(pred))


X_s6 = pd.get_dummies(bnb[["accommodates", "dist_km", "room_type", "district"]], dtype=int)   # Session 6 features
models = {"linear, Session 6 features": LinearRegression(),
          "ridge": make_pipeline(prep, Ridge(alpha=10)),
          "random forest": make_pipeline(prep, RandomForestRegressor(200, min_samples_leaf=3, max_features=0.3,
                                                                     n_jobs=-1, random_state=0)),
          "gradient boosting": make_pipeline(prep, HistGradientBoostingRegressor(random_state=0))}
rows = []
for name, model in models.items():
    Xm = X_s6 if name.startswith("linear") else X
    r2_rand, mae_rand = euro_scores(model, Xm, KFold(5, shuffle=True, random_state=0))
    r2_grp, mae_grp = euro_scores(model, Xm, GroupKFold(5), groups=hosts)
    rows.append({"model": name, "R² random": r2_rand, "R² by host": r2_grp,
                 "MAE € random": mae_rand, "MAE € by host": mae_grp})
pd.DataFrame(rows).round(3)

**Task A1.** (1) For which model is the gap between random and grouped validation largest, and for which is it almost zero? Why would a flexible model with many features profit most from seeing other flats of the same host? (2) For which use is the random estimate acceptable, and for which use do you need the grouped one?

### A2 · A held-out test set of hosts and a bootstrap interval

We lock away 20 % of the hosts (with all their listings), fit the ridge model on the rest, and estimate the MAE in euros on the locked hosts once. Two bootstrap intervals: resampling **listings** (assumes independent rows) and resampling **hosts** (a cluster bootstrap that keeps each host's listings together).

In [ ]:
dev, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X, y, groups=hosts))
ridge = make_pipeline(prep, Ridge(alpha=10)).fit(X.iloc[dev], y.iloc[dev])
price_test = bnb["price"].iloc[test].to_numpy()
pred_test = np.exp(ridge.predict(X.iloc[test]))
abs_err = np.abs(price_test - pred_test)
print(len(test), "test listings of", hosts.iloc[test].nunique(), "hosts; MAE €", round(abs_err.mean(), 1),
      "; median absolute error €", round(np.median(abs_err), 1))

rng = np.random.default_rng(0)
boot_rows = [abs_err[rng.integers(0, len(abs_err), len(abs_err))].mean() for _ in range(1000)]
test_hosts = hosts.iloc[test].to_numpy()
by_host = pd.Series(abs_err).groupby(test_hosts)
sums, counts = by_host.sum().to_numpy(), by_host.size().to_numpy()
boot_hosts = []
for _ in range(1000):
    pick = rng.integers(0, len(sums), len(sums))             # draw hosts with replacement
    boot_hosts.append(sums[pick].sum() / counts[pick].sum())
print("95% interval, listings resampled:", np.percentile(boot_rows, [2.5, 97.5]).round(1))
print("95% interval, hosts resampled:   ", np.percentile(boot_hosts, [2.5, 97.5]).round(1))

**Task A2.** Which interval is wider, and why is it the honest one here? Express the MAE as a share of the median price of the test listings. Would you trust the model to suggest a price for a new host?

## Part B · Regularisation and tuning

### B1 · Ridge and lasso on 480 columns

In [ ]:
cv = GroupKFold(5)
for name, reg in [("OLS", LinearRegression()), ("ridge a=10", Ridge(alpha=10)),
                  ("lasso a=0.001", Lasso(alpha=0.001, max_iter=5000)), ("lasso a=0.01", Lasso(alpha=0.01, max_iter=5000))]:
    m = make_pipeline(prep, reg).fit(X, y)
    coef = m[-1].coef_
    r2 = r2_score(y, cross_val_predict(m, X, y, cv=cv, groups=hosts))
    print(f"{name:13s} CV R² {r2:.3f}  largest |b| {np.abs(coef).max():.2f}  non-zero {(coef != 0).sum()}")

names = m[0].get_feature_names_out()
lasso = make_pipeline(prep, Lasso(alpha=0.01, max_iter=5000)).fit(X, y)
b = pd.Series(lasso[-1].coef_, index=names)
b = b[b != 0].sort_values()
print("\nlargest negative and positive coefficients of the lasso (log scale):")
print(pd.concat([b.head(5), b.tail(8)]).round(3).to_string())

**Task B1.** Read the lasso coefficients as percentages (`np.exp(b) - 1`). Which kept features make sense to a host? Which ones are proxies (for example an amenity that mainly cheaper or more expensive flats list)? Why must these coefficients not be read as "adding a dishwasher raises the price by x %"?

### B2 · Validation curves: when does the penalty matter?

In [ ]:
alphas = np.logspace(-2, 4, 13)
small = np.random.default_rng(0).choice(len(X), 1000, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (label, rows) in zip(axes, [("all listings", np.arange(len(X))), ("1,000 listings", small)]):
    tr, va = validation_curve(make_pipeline(prep, Ridge()), X.iloc[rows], y.iloc[rows], groups=hosts.iloc[rows],
                              param_name="ridge__alpha", param_range=alphas, cv=cv, scoring="r2")
    ax.semilogx(alphas, tr.mean(1), marker="o", label="training")
    ax.semilogx(alphas, va.mean(1), marker="o", label="validation")
    ax.set_title(f"{label}: best alpha {alphas[va.mean(1).argmax()]:g}")
    ax.set_xlabel("ridge alpha")
axes[0].set_ylabel("R² (log price)")
axes[0].legend()
plt.show()

### B3 · Learning curves: would more listings help?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (label, est) in zip(axes, [("ridge", make_pipeline(prep, Ridge(alpha=10))),
                                   ("gradient boosting", make_pipeline(prep, HistGradientBoostingRegressor(random_state=0)))]):
    sizes, tr, va = learning_curve(est, X, y, groups=hosts, cv=cv, train_sizes=np.linspace(0.1, 1, 6), scoring="r2", n_jobs=-1)
    ax.plot(sizes, tr.mean(1), marker="o", label="training")
    ax.plot(sizes, va.mean(1), marker="o", label="validation")
    ax.set_title(label)
    ax.set_xlabel("training listings")
axes[0].set_ylabel("R² (log price)")
axes[0].legend()
plt.show()

**Task B2–B3.** Diagnose each panel: underfitting, overfitting or neither? For the 1,000-listing case, how much validation R² does the best penalty buy compared with almost no penalty? For gradient boosting, what does the learning curve suggest about collecting more data?

### B4 · Grid search and randomised search, with random and grouped folds

In [ ]:
for label, splitter in [("random KFold", KFold(5, shuffle=True, random_state=0)), ("GroupKFold by host", cv)]:
    grid = GridSearchCV(make_pipeline(prep, Ridge()), {"ridge__alpha": [0.1, 1, 3, 10, 30, 100, 300]},
                        cv=splitter, scoring="r2")
    grid.fit(X, y, groups=hosts if label.startswith("Group") else None)
    print(f"{label:20s} best alpha {grid.best_params_['ridge__alpha']:>5}  CV R² {grid.best_score_:.3f}")

In [ ]:
space = {"histgradientboostingregressor__learning_rate": loguniform(0.01, 0.3),
         "histgradientboostingregressor__max_leaf_nodes": randint(4, 64),
         "histgradientboostingregressor__min_samples_leaf": randint(5, 100),
         "histgradientboostingregressor__l2_regularization": loguniform(1e-3, 10)}
rand = RandomizedSearchCV(make_pipeline(prep, HistGradientBoostingRegressor(random_state=0)), space,
                          n_iter=15, cv=cv, scoring="r2", random_state=0, n_jobs=-1)
rand.fit(X, y, groups=hosts)
res = pd.DataFrame(rand.cv_results_).sort_values("rank_test_score")
cols = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
res[cols].rename(columns=lambda c: c.replace("param_histgradientboostingregressor__", "")).round(3).head(8)

**Task B4.** (1) Do random and grouped folds choose the same `alpha`? Do they report the same score? Which score would you put in a report for new hosts? (2) In the randomised search, how many settings lie within one standard deviation of the best? Is the best setting clearly better than the defaults (CV R² of the default model in A1)? (3) Why is `best_score_` an optimistic estimate, and how would you get an unbiased one (theory page 03)?

## Part C · Two leaking workflows and their fix

A tempting idea for a price model: "listings in an expensive neighbourhood are expensive, and a host with expensive flats has expensive flats", so add the **mean log price of the neighbourhood** and the **mean log price of the host** as features. Computed with a `groupby` on all listings, both columns contain the listing's own price. That is the first leaking workflow: a preprocessing step that looks at the target, fitted outside the cross-validation. The second is subtler: even when the means are learned inside the pipeline, random folds put other listings of the same host into training, which a first-time host never has.

We use a smaller feature set than in Parts A–B (no amenities) and compare ridge and gradient boosting, each with random and host-grouped folds.

In [ ]:
from sklearn.preprocessing import TargetEncoder

num_c = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "minimum_nights",
         "availability_365", "number_of_reviews", "review_scores_rating"]
cat_c = ["room_type", "district", "property_type"]
Xc = bnb[num_c + cat_c + ["neighbourhood"]].assign(host=hosts.astype(str))
# the two "convenient" columns, computed from ALL listings (each listing's own price included)
Xc["nb_price"] = y.groupby(Xc["neighbourhood"]).transform("mean")
Xc["host_price"] = y.groupby(Xc["host"]).transform("mean")
print("share of listings whose host has only this listing:", round((n_per_host == 1).mean(), 3))


def num_prep():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())


def onehot():
    return OneHotEncoder(handle_unknown="ignore", sparse_output=False)   # dense, for gradient boosting


def encoder():
    return TargetEncoder(cv=KFold(5, shuffle=True, random_state=0))      # learns the means inside each training part


workflows = {
    "means from all rows": make_column_transformer((num_prep(), num_c + ["nb_price", "host_price"]), (onehot(), cat_c)),
    "TargetEncoder in Pipeline": make_column_transformer((num_prep(), num_c), (onehot(), cat_c),
                                                         (encoder(), ["neighbourhood", "host"])),
    "TargetEncoder, no host column": make_column_transformer((num_prep(), num_c), (onehot(), cat_c),
                                                             (encoder(), ["neighbourhood"])),
}
regressors = {"ridge": Ridge(alpha=10), "gradient boosting": HistGradientBoostingRegressor(random_state=0)}
splitters = {"random": (KFold(5, shuffle=True, random_state=0), None), "by host": (GroupKFold(5), hosts)}

### C1 · What cross-validation reports

In [ ]:
rows = []
for reg_name, reg in regressors.items():
    for wf_name, prep_c in workflows.items():
        for sp_name, (splitter, groups) in splitters.items():
            pred = cross_val_predict(make_pipeline(prep_c, reg), Xc, y, cv=splitter, groups=groups, n_jobs=-1)
            rows.append({"model": reg_name, "workflow": wf_name, "folds": sp_name,
                         "R²": r2_score(y, pred), "MAE €": mean_absolute_error(bnb["price"], np.exp(pred))})
leak_table = pd.DataFrame(rows)
leak_table.pivot_table(index=["model", "workflow"], columns="folds", values=["R²", "MAE €"]).round(3)

### C2 · What happens with new hosts

The check that decides: lock away 20 % of the hosts, fit each workflow on the other hosts, and predict the locked-away listings the way the model would be used. A new host has no price history, so the host mean is missing (the imputer fills in the median); the neighbourhood means come from the development hosts only.

In [ ]:
dev_c, test_c = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(Xc, y, groups=hosts))
X_dev = Xc.iloc[dev_c].copy()
X_dev["nb_price"] = y.iloc[dev_c].groupby(X_dev["neighbourhood"]).transform("mean")
X_dev["host_price"] = y.iloc[dev_c].groupby(X_dev["host"]).transform("mean")
X_new = Xc.iloc[test_c].copy()
X_new["nb_price"] = X_new["neighbourhood"].map(y.iloc[dev_c].groupby(X_dev["neighbourhood"]).mean())
X_new["host_price"] = np.nan                                    # a new host has no price history

rows = []
for reg_name, reg in regressors.items():
    for wf_name, prep_c in workflows.items():
        pred = make_pipeline(prep_c, reg).fit(X_dev, y.iloc[dev_c]).predict(X_new)
        rows.append({"model": reg_name, "workflow": wf_name, "R² new hosts": r2_score(y.iloc[test_c], pred),
                     "MAE € new hosts": mean_absolute_error(bnb["price"].iloc[test_c], np.exp(pred))})
pd.DataFrame(rows).round(3)

In our run the means from all rows promised an MAE of €26 to €28 for both models and both kinds of folds; on new hosts the same workflows were off by €69 (ridge) and €65 (gradient boosting), clearly *worse* than the honest models (about €52 and €50): they had learned to rely on a column that a new host does not have. The pipeline with random folds promised €48 (ridge) and €44 (gradient boosting); only the pipeline with host-grouped folds (€52 and €50) matched what happened on new hosts.

**Task C.** (1) Why do host-grouped folds not repair the first workflow? (2) Put the cross-validated MAE of each workflow next to its MAE on new hosts. Which estimate would you report to a host who lists a first flat? (3) Once both leaks are fixed, does the host column help? Explain why an unknown host gets the same value as every other unknown host. (4) Name one more column of `listings.parquet` that would leak the price into a price model (look at the `estimated_*` columns; Session 9 returns to them).

**Team project.** If your project data have groups (customers, stores, hosts, patients) or time, write down which splitter your validation plan uses and why, and list every feature that is computed from the target or from other rows.